# 习题数值复算

按题号查找复算单元。首次使用时先运行安装单元（已有 `mpmath` 可跳过），再运行公共设置；之后可任选题号独立复算。

In [ ]:
%pip install mpmath

In [ ]:
import mpmath as mp

mp.mp.dps = 40
theta = mp.siegeltheta

def theta_prime(t):
    z = mp.mpf(1)/4 + mp.j*t/2
    return (mp.re(mp.digamma(z)) - mp.log(mp.pi))/2

## 7.4

对照正整数阶乘的 Stirling 近似，并在 $t=1,10$ 比较 Gamma 模平方及其主项。

In [ ]:
for n in (1, 5, 10, 20):
    f = mp.factorial(n)
    approx = mp.sqrt(2*mp.pi*n)*(n/mp.e)**n
    print(n, mp.nstr(f, 8), mp.nstr(f/approx, 10), mp.nstr(1+1/mp.mpf(12*n), 10))
for t in (1, 10):
    main = 2*mp.pi*mp.exp(-mp.pi*t)
    for a, lead in [(mp.mpf('0.5'), main), (0, main/t)]:
        value = abs(mp.gamma(a+mp.j*t))**2
        print(t, a, mp.nstr(value, 10), mp.nstr(lead, 10))

## 9.1

复算 $N=10,100,1000$ 的交错部分和与相邻平均；最后输出由 $\eta(1/2)$ 换算的 $\zeta(1/2)$。

In [ ]:
for N in (10, 100, 1000):
    P = mp.fsum((-1)**(k-1)/mp.sqrt(k) for k in range(1, N+1))
    A = P + (-1)**N/(2*mp.sqrt(N+1))
    print(N, mp.nstr(P, 13), mp.nstr(A, 13))
print(mp.nstr(mp.altzeta(mp.mpf('0.5'))/(1-mp.sqrt(2)), 13))

## 9.4(c)

按解答表中的四组参数比较 3–4–1 乘积与两因子乘积；检查两列均大于 1。

In [ ]:
for sigma in map(mp.mpf, ('1.05', '1.01')):
    for t in map(mp.mpf, ('1', '14.134725')):
        a = mp.zeta(sigma)
        b = abs(mp.zeta(sigma+mp.j*t))
        c = abs(mp.zeta(sigma+2*mp.j*t))
        print(sigma, t, mp.nstr(a**3*b**4*c, 10), mp.nstr(a*b, 10))

## 9.7(b)

将积分从 1 分段积到无穷远，保留 theta 和式的前八项；用积分值复算两个 zeta 特值。

In [ ]:
for s in (mp.mpf(2), mp.mpf('0.5')):
    def term(x):
        omega = mp.fsum(mp.exp(-mp.pi*k*k*x) for k in range(1, 9))
        return omega*(x**(s/2-1)+x**(-(s+1)/2))
    H = mp.quad(term, [1, 2, 5, mp.inf])
    zeta_value = (1/(s*(s-1))+H)*mp.pi**(s/2)/mp.gamma(s/2)
    print(s, mp.nstr(H, 12), mp.nstr(zeta_value, 12))

## 19.1(a) 相位的最低点与左支上相位等于 $-\pi$ 的点

In [ ]:
t_min = mp.findroot(theta_prime, (6, 7))
t_left = mp.findroot(lambda t: theta(t) + mp.pi, mp.mpf('3.4'),
                     solver='newton', df=theta_prime)
assert 0 < t_left < t_min
for label, value in [('t_min', t_min), ('theta(t_min)', theta(t_min)), ('t_left', t_left)]:
    print(label, '=', mp.nstr(value, 12))

## 19.1(b) 右支 Gram 点与 Hardy Z 值

Newton 迭代使用相位导数。表中最后一列为 $(-1)^n Z(g_n)$。

In [ ]:
guess = mp.mpf('9.7')
print(' n          g_n           Z(g_n)       (-1)^n Z(g_n)')
for n in range(-1, 6):
    g = mp.findroot(lambda t: theta(t) - n*mp.pi, guess,
                    solver='newton', df=theta_prime)
    assert g > 7 and theta_prime(g) > 0
    Z = mp.siegelz(g)
    signed_Z = Z if n % 2 == 0 else -Z
    print(f'{n:2d}  {mp.nstr(g, 12):>14}  {mp.nstr(Z, 10):>13}  {mp.nstr(signed_Z, 10):>15}')
    guess = g + 2*mp.pi/mp.log(g/(2*mp.pi))